# HuBERT-ECG ince ayarı — domates sulama stresi (Colab)

**Başlamadan önce:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç, kaydet.

Sonra hücreleri sırayla çalıştır (**Shift + Enter**) ya da **Çalışma zamanı → Tümünü çalıştır**.

- Sonuçlar Google Drive'da `bitkiEkg_sonuclar/` klasörüne, her bitki için ayrı dosya olarak yazılır.
- Oturum koparsa not defterini baştan çalıştırman yeterli: biten bitkiler atlanır, kaldığı yerden devam eder.
- Ayarlar deney öncesinde sabitlenmiştir (`src/bitki_ekg/ince_ayar.py`); değiştirme.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç, sonra bu hücreyi tekrar çalıştır.')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive (sonuçlar buraya kaydedilir)

Açılan pencerede Google hesabınla izin ver.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
CIKTI = Path('/content/drive/MyDrive/bitkiEkg_sonuclar')
CIKTI.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', CIKTI)

## 3. Kod ve veri (GitHub'dan)

In [ ]:
!rm -rf /content/bitkiEkg
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!pip install -q "transformers>=4.40,<5" safetensors huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
import transformers
print('transformers', transformers.__version__)
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 yüklü: Çalışma zamanı → Yeniden başlat, sonra baştan çalıştır.'

In [ ]:
import numpy as np
from bitki_ekg.ekg_girdi import hubert_girdisi
from bitki_ekg.ince_ayar import AYARLAR, lopo_calistir

def veri_yukle(pencere):
    z = np.load(f'/content/bitkiEkg/data/colab/domates_ikili_{pencere}.npz')
    X = hubert_girdisi(z['Z'].astype(np.float32))
    return X, z['y'].astype(int), z['bitki'].astype(int)

X, y, g = veri_yukle('1h')
print('Girdi:', X.shape, '| sınıflar:', np.bincount(y), '| bitkiler:', sorted(set(g)))
print('Ayarlar:', AYARLAR)

## 4. Eğitim — 1 saatlik pencere

İki deney × 12 bitki. İlk bitkinin süresine bakarak toplam süreyi tahmin edebilirsin (beklenen: bitki başına birkaç dakika).

In [ ]:
DENEYLER = [('onceden_egitilmis', True), ('rastgele_baslatilmis', False)]
for ad, onceden in DENEYLER:
    print(f'=== {ad} ===', flush=True)
    lopo_calistir(X, y, g, onceden, CIKTI / '1h' / ad, 'cuda')

## 5. Sonuç özeti

In [ ]:
import json
import pandas as pd

def ozet(pencere):
    satir = []
    for ad, _ in DENEYLER:
        for f in sorted((CIKTI / pencere / ad).glob('bitki_*.json')):
            s = json.loads(f.read_text(encoding='utf-8'))
            satir.append({'deney': ad, 'bitki': s['test_bitkisi'], 'dogruluk': s['dogruluk'],
                          'f1': s['f1'], 'auc': s['auc'], 'en_iyi_epoch': s['en_iyi_epoch']})
    df = pd.DataFrame(satir)
    print(f'[{pencere}] tamamlanan bitki sayısı:', df.groupby('deney').size().to_dict())
    print((df.groupby('deney')[['dogruluk', 'f1', 'auc']].agg(['mean', 'std']) * 100).round(1))
    print('Karşılaştırma (LOPO doğruluk): LightGBM 72,9 · MiniRocket 72,5 (1 sa) / 70,5 · 73,1 (30 dk)')
    df.to_csv(CIKTI / f'ozet_{pencere}.csv', index=False)
    return df

ozet('1h')

## 6. (İsteğe bağlı) 30 dakikalık pencere

1 saatlik deney bittikten sonra GPU süren kaldıysa çalıştır; yaklaşık iki katı sürer.

In [ ]:
X, y, g = veri_yukle('30min')
for ad, onceden in DENEYLER:
    print(f'=== {ad} ===', flush=True)
    lopo_calistir(X, y, g, onceden, CIKTI / '30min' / ad, 'cuda')
ozet('30min')